# 1 · Modelo de referencia

**Proyecto:** Predicción de readmisión hospitalaria en pacientes diabéticos
**Dataset:** *Diabetes 130-US hospitals for years 1999-2008* (UCI Machine Learning Repository)
**Autor:** Diego Rodríguez Díaz del Campo

---

### Objetivo de este notebook

Fijar las reglas con las que se compararán todos los modelos del proyecto, es decir, cómo se valida y qué
nota hay que superar, y obtener el primer resultado con un modelo simple: la regresión logística.

El modelo debe **ordenar los ingresos por riesgo de readmisión a 30 días** en el momento del alta, para
priorizar el seguimiento. Se mide con **PR-AUC** como métrica principal, **ROC-AUC** como complementaria y
recall y precisión en el **20 % de ingresos de mayor riesgo**.

| | |
|---|---|
| **Entrada** | `data/train.csv` (79.433 × 160), salida del notebook 04 de [readmision_diabetes_eda](https://github.com/diieeguiitooo/readmision_diabetes_eda) |
| **Variable objetivo** | `readmitted_30d` (1 = reingreso antes de 30 días) |
| **Salida** | Resultados de validación de la regresión logística: la referencia para los notebooks 02 y 03 |

`data/test.csv` no se usa en este notebook: queda reservado para la evaluación final (notebook 03).

### Contenido

1. Carga y comprobaciones
2. Separación de predictoras, objetivo y grupos
3. Diseño de la validación
4. Referencias que hay que superar
5. Regresión logística
6. Conclusiones

## 1 · Carga y comprobaciones

Se parte de `data/train.csv`, descargado con `scripts/descargar_datos.py` desde un commit fijo de
`readmision_diabetes_eda`. El script ya garantiza que el archivo es idéntico byte a byte al que guardó el
notebook 04 (huella SHA-256); lo que falta comprobar es que pandas lo ha leído como se espera.

#### Decisiones

- **Solo se carga `train.csv`.** `test.csv` no se abre hasta el notebook 03: cualquier decisión tomada
  mirándolo lo convertiría en parte del entrenamiento, y su nota final dejaría de medir el rendimiento con
  datos nuevos.
- **`read_csv` sin opciones.** Las 160 columnas son enteros, así que no reaparece el problema de tipos de los
  `*_id` ni el aviso de `payer_code` del proyecto EDA.

### Qué se comprueba

Cada comprobación se compara con el valor que dejó el notebook 04:

| Comprobación | Esperado | Qué descarta |
|---|---|---|
| Dimensiones | 79.433 × 160 | Filas perdidas o una columna de índice añadida (`Unnamed: 0`) |
| Tipos | 160 columnas `int64` | Una columna leída como texto |
| Nulos | 0 | Valores vacíos, que la regresión logística no admite |
| Objetivo | 70.362 ceros y 9.071 unos (11,42 %) | Un objetivo distinto del preparado |
| Pacientes distintos | 55.992 | Grupos que no cuadran con la partición del notebook 04 |

In [1]:
# Este apartado solo carga y comprueba el CSV: basta con pandas y pathlib.
# scikit-learn se importa en cada apartado donde se usa por primera vez.
import pandas as pd
from pathlib import Path

# El notebook se ejecuta desde notebooks/, así que la raíz del proyecto es la carpeta padre.
# Construir las rutas desde ahí evita depender de dónde esté clonado el repo.
BASE = Path.cwd().parent
DATOS = BASE / 'data'
RUTA_TRAIN = DATOS / 'train.csv'

# Si el archivo no está, falla aquí con un mensaje claro y no más adelante dentro de read_csv
assert RUTA_TRAIN.exists(), f'No se encuentra {RUTA_TRAIN}: ejecuta scripts/descargar_datos.py'
print(f'Raíz del proyecto: {BASE}')

Raíz del proyecto: /Users/diegorodriguezdiazdelcampo/Cosas Diego/IA/Proyectos personales/Model Diabetes


In [2]:
df = pd.read_csv(RUTA_TRAIN)

# Las cinco comprobaciones de la tabla, en el mismo orden
print(f'Dimensiones:          {df.shape}')
print(f'Tipos:                {df.dtypes.astype(str).value_counts().to_dict()}')
print(f'Nulos:                {df.isna().sum().sum()}')

# Recuento por clase, no solo cuántas clases hay: así se comparan con los 70.362 / 9.071 del NB04
conteo_objetivo = df['readmitted_30d'].value_counts().sort_index()
print(f'Objetivo (0 / 1):     {conteo_objetivo[0]} / {conteo_objetivo[1]} '
      f'({df["readmitted_30d"].mean():.2%} positivos)')
print(f'Pacientes distintos:  {df["patient_nbr"].nunique()}')

Dimensiones:          (79433, 160)
Tipos:                {'int64': 160}
Nulos:                0
Objetivo (0 / 1):     70362 / 9071 (11.42% positivos)
Pacientes distintos:  55992


### Interpretación del apartado 1

Las cinco comprobaciones coinciden con lo que dejó el notebook 04:

| Comprobación | Esperado | Obtenido |
|---|---|---|
| Dimensiones | 79.433 × 160 | 79.433 × 160 |
| Tipos | 160 columnas `int64` | 160 columnas `int64` |
| Nulos | 0 | 0 |
| Objetivo | 70.362 ceros y 9.071 unos (11,42 %) | 70.362 ceros y 9.071 unos (11,42 %) |
| Pacientes distintos | 55.992 | 55.992 |

Las comprobaciones de tipos y de nulos no son independientes: una columna `int64` no puede contener
`NaN`, que es un valor decimal, así que 160 columnas enteras implican 0 nulos. Se mantienen las dos
porque la de nulos se lee sin conocer esa regla.

El archivo se ha leído como se esperaba y se puede pasar a separar sus columnas.

## 2 · Separación de predictoras, objetivo y grupos

scikit-learn recibe por separado la matriz de predictoras (`X`) y el vector objetivo (`y`). La
validación agrupada del apartado 3 necesita además un tercer vector, `grupos`, que indica a qué
paciente pertenece cada ingreso.

#### Decisiones

- **`y` es `readmitted_30d`.** Está en la posición 12, no al final, así que se selecciona por nombre y
  no por posición.
- **`patient_nbr` sale de `X` y se guarda como `grupos`.** Un identificador no tiene significado
  clínico y el modelo podría aprender patrones espurios a partir del número. Solo sirve para que todos
  los ingresos de un mismo paciente caigan en el mismo pliegue.
- **`X` son las 158 columnas restantes, sin escalar.** El escalado va dentro del pipeline (apartado 5)
  y se aprende solo con los datos de entrenamiento de cada pliegue.

### Qué se comprueba

| Comprobación | Esperado |
|---|---|
| Dimensiones de `X` | 79.433 × 158 |
| Longitud de `y` y de `grupos` | 79.433 cada uno |
| `readmitted_30d` y `patient_nbr` en `X` | Ninguna de las dos |

In [3]:
# Objetivo y grupos se seleccionan por nombre; X es todo lo demás
X = df.drop(columns=['readmitted_30d', 'patient_nbr'])
y = df['readmitted_30d']
grupos = df['patient_nbr']

# Las tres comprobaciones de la tabla
print(f'Dimensiones de X:     {X.shape}')
print(f'Longitud de y:        {len(y)}')
print(f'Longitud de grupos:   {len(grupos)}')
print(f'Objetivo en X:        {"readmitted_30d" in X.columns}')
print(f'patient_nbr en X:     {"patient_nbr" in X.columns}')

Dimensiones de X:     (79433, 158)
Longitud de y:        79433
Longitud de grupos:   79433
Objetivo en X:        False
patient_nbr en X:     False


### Interpretación del apartado 2

| Comprobación | Esperado | Obtenido |
|---|---|---|
| Dimensiones de `X` | 79.433 × 158 | 79.433 × 158 |
| Longitud de `y` y de `grupos` | 79.433 cada uno | 79.433 y 79.433 |
| `readmitted_30d` y `patient_nbr` en `X` | Ninguna de las dos | Ninguna (`False` y `False`) |

`X`, `y` y `grupos` tienen las mismas filas y en el mismo orden, porque los tres salen de `df` sin
reordenar. El ingreso de la fila *i* de `X` tiene su objetivo en la fila *i* de `y` y su paciente en la
fila *i* de `grupos`, que es lo que necesita la validación del apartado 3.